# Fine-tuning Qwen2.5-Coder-1.5B to write shell commands

This notebook produces the model the app uses to turn a plain-English request
into one bash command. It fine-tunes
[Qwen2.5-Coder-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct)
with QLoRA on about 49,000 request and command pairs, then exports it as a
4-bit GGUF file for llama.cpp.

| Step | What happens |
|---|---|
| Data (section 3) | Four public datasets are pooled, each pinned to a commit, then filtered for wrong answers, synthetic tasks, test-set leakage and rarely used tools |
| Training (section 4) | Two epochs, evaluated and checkpointed every 250 steps; the checkpoint with the lowest validation loss is kept |
| Export (section 5) | The fine-tuned model and the untouched base model go through the same GGUF export, so the two files differ only in their weights |
| Records (section 6) | The exact training data, a manifest, the loss log and package versions are saved next to the model |

The base model is exported as well so the two can be compared fairly; the
tests are in `evaluation/` in the project.

**Runtime.** Colab with a T4 GPU: about 5 minutes to prepare the data and
about 2.5 hours to train with the default settings.
Checkpoints go to Drive; if the session disconnects, run every cell again
and training resumes from the last checkpoint.

## 1. Setup

In [ ]:
%%capture
# The Unsloth release this notebook was run with.
!pip install "unsloth==2026.11.2"

In [ ]:
# Unsloth must be imported before trl and transformers so it can patch them.
import unsloth

import gc
import hashlib
import importlib.metadata
import inspect
import json
import re
import shutil
import subprocess
import time
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset
from huggingface_hub import hf_hub_download

print(torch.cuda.get_device_name(0))

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/ghost-in-the-shell/finetune")
except ImportError:
    OUTPUT_DIR = Path("/kaggle/working/finetune")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(OUTPUT_DIR)

## 2. Configuration

Dataset revisions are pinned to the commits that were inspected, so a later
upload cannot change what the model is trained on.

In [ ]:
NAME = "qwen2.5-coder-1.5b"
BASE_MODEL = "unsloth/Qwen2.5-Coder-1.5B-Instruct"

# The prompt the app sends with every request (src/ghost_in_the_shell/model.py).
SYSTEM_PROMPT = (
    "You translate natural-language requests into a single Linux bash command. "
    "Reply with only the command on one line: no explanation, no markdown, no code fences."
)

# name -> (Hugging Face dataset, pinned commit)
SOURCES = {
    "frost": ("Frost2o24/bash-instruct-III-55k", "2a9ec4c4ab57a03857078e20df843b558b090d32"),      # MIT
    "mecha": ("mecha-org/linux-command-dataset", "d3883ec38c0458a73a0b8aef19103cb2d006c87c"),      # Apache-2.0
    "aelhalili": ("aelhalili/bash-commands-dataset", "67a539a9c6358574fe4f22e126cba3421fff4645"),  # MIT
    "alfa": ("westenfelder/NL2SH-ALFA", "a99cb5784cf5c2a42b1cc26c1903d9c3b35206ba"),               # MIT
    "dale": ("manuelgscom/dale-bash-test-v2", "7edf49aab21a0b605892f21bed25b0ae17a23126"),         # CC-BY, test only
}

SEED = 42
VAL_COMMANDS = 1_000        # distinct commands held out for validation
MAX_TRAIN = None            # set a number for a quick trial run
MAX_COMMAND_LENGTH = 200
MIN_PER_UTILITY = 20        # a tool needs this many examples, or to appear in two sources
MAX_UTILITY_SHARE = 0.10    # no tool may exceed about this share of the data

MAX_SEQ_LENGTH = 512
LORA = dict(r=16, lora_alpha=16, lora_dropout=0.0)
BATCH_SIZE, GRAD_ACCUM = 8, 2                           # effective batch size 16
EPOCHS = 2
LEARNING_RATE = 2e-4
WARMUP_SHARE = 0.03
EVAL_EVERY = 250            # steps; also the checkpoint interval
PATIENCE = 4                # stop after this many evaluations without improvement

## 3. Training data

Four sources are pooled, most trusted first, so that when two sources contain
the same request the more trusted pair is the one kept:

1. **Bash Instruct III**: synthetic pairs its authors validated by execution.
   Multi-line scripts are dropped, and one answer is kept per request.
2. **mecha-org** and **aelhalili**: short everyday requests.
3. **NL2SH-ALFA**: the largest source, but unverified.

Two test sets must stay unseen: the NL2SH-ALFA test set (the public benchmark) and dale-bash-test-v2 (a second held-out set).

In [ ]:
def fetch(source, file):
    repo, revision = SOURCES[source]
    return hf_hub_download(repo, file, repo_type="dataset", revision=revision)

def pairs(source, requests, commands):
    return pd.DataFrame({"nl": list(requests), "bash": list(commands), "source": source})

frost = pd.read_json(fetch("frost", "bash_dataset.jsonl"), lines=True)
frost = frost[frost.category != "script"].drop_duplicates("variant_group")
mecha = pd.read_json(fetch("mecha", "linuxcommands.json"))
aelhalili = pd.read_json(fetch("aelhalili", "dataset.json"))
alfa = pd.read_csv(fetch("alfa", "train.csv"))

pool = pd.concat([
    pairs("frost", frost.messages.map(lambda m: m[1]["content"]), frost.messages.map(lambda m: m[2]["content"])),
    pairs("mecha", mecha.input, mecha.output),
    pairs("aelhalili", aelhalili.prompt, aelhalili.response),
    pairs("alfa", alfa.nl, alfa.bash),
], ignore_index=True).dropna()
pool["nl"], pool["bash"] = pool.nl.str.strip(), pool.bash.str.strip()

alfa_test = pd.read_csv(fetch("alfa", "test.csv"))
dale_test = pd.read_json(fetch("dale", "test.jsonl"), lines=True)
test_requests = pd.concat([alfa_test.nl, dale_test.request])
test_commands = pd.concat([alfa_test.bash, alfa_test.bash2, dale_test.reference_commands.explode()]).dropna()

print(pool.source.value_counts().to_string())

### Filters

Each filter records how many pairs it removed; the table is printed at the end
and saved in the manifest.

- **Format.** One line, at most 200 characters, plain-ASCII request, no
  unfilled placeholders such as `{{file}}` or `/path/to`.
- **Random tasks.** A tenth of NL2SH-ALFA is machine-made "pick a random
  file" tasks that no user asks for. This also drops the few genuine uses of
  `shuf`.
- **Wrong scope.** The request says "current directory" but the command
  searches `/` or `~` (15% of such requests in NL2SH-ALFA).
- **Catastrophic commands.** Recursive deletes or permission changes on `/`
  or `~`, writing to disk devices, piping a download into a shell. Ordinary
  `rm`, `sudo` and `shutdown` stay: the app has its own safety check, and
  the model still has to answer "delete app.log".
- **Test leakage.** Any pair whose request or command matches a test item
  after normalisation, or whose request shares 80% of its words with a test
  request.
- **Duplicates**, then **syntax** (`bash -n`).
- **Tool balance.** NL2SH-ALFA has thousands of tools with a handful of
  examples each (`gnmic`, `bundletool`), too few for a 1.5B model to learn.
  A tool is kept if it has at least 20 examples or appears in two sources,
  which keeps everyday tools such as `hostname` that are simply short on
  examples. Each remaining tool is then capped, which mostly trims `find`
  (29% of NL2SH-ALFA). Both rules look only at the training pool, never at
  the test sets.

In [ ]:
def normalise(text):
    return re.sub(r"\W+", " ", str(text).lower()).strip()

def words(text):
    return set(normalise(text).split())

def utility(command):
    # The program a command runs, skipping sudo and leading VAR=value assignments.
    tokens = command.split()
    while tokens and (tokens[0] == "sudo" or re.match(r"^\w+=", tokens[0])):
        tokens = tokens[1:]
    return tokens[0].lstrip("`($") if tokens else ""

def valid_bash(command):
    return subprocess.run(["bash", "-n", "-c", command], capture_output=True).returncode == 0

CATASTROPHIC = "|".join([
    r"\brm\b[^|;&]*\s-\w*[rR]\w*[^|;&]*\s(/|/\*|~|~/|~/\*|\$HOME|\*)(\s|$)",   # recursive rm on / ~ or *
    r"\bfind\s+(/|~|\$HOME)\s[^|;&]*(-delete|-exec\s+rm)",                      # find-and-delete from / or ~
    r"\b(chmod|chown)\b[^|;&]*-\w*R\w*[^|;&]*\s(/|~|\$HOME)(\s|$)",             # recursive chmod/chown on / or ~
    r"\bdd\b[^|;&]*\bof=/dev/(sd|nvme|hd|vd|mmcblk)|>\s*/dev/(sd|nvme|hd|vd)",  # write to a disk device
    r"\bmkfs(\.\w+)?\b|\b(wipefs|shred)\b[^|;&]*/dev/",                         # format or wipe a disk
    r"\b(curl|wget)\b[^|;&]*\|\s*(sudo\s+)?(ba|z)?sh\b",                        # pipe a download into a shell
    r":\(\)\s*\{|\bkill\s+-9\s+-1\b",                                           # fork bomb, kill everything
])

test_request_set = set(test_requests.map(normalise))
test_command_set = set(test_commands.map(normalise))
test_word_sets = [words(t) for t in test_requests]

def near_test_request(text):
    a = words(text)
    return any(len(a & b) >= 0.8 * len(a | b) for b in test_word_sets
               if 0.8 * len(b) <= len(a) <= 1.25 * len(b))   # sizes too different cannot reach 80%

report = []

def keep(df, name, mask):
    report.append({"filter": name, "removed": int((~mask).sum()), "left": int(mask.sum())})
    return df[mask]

In [ ]:
clean = pool
clean = keep(clean, "format",
             ~clean.bash.str.contains("\n") & clean.bash.str.len().between(1, MAX_COMMAND_LENGTH)
             & clean.nl.map(str.isascii) & (clean.nl.str.len() >= 8)
             & ~clean.bash.str.contains(r"\{\{|<[A-Za-z_ ]+>|/path/to"))
clean = keep(clean, "random tasks",
             ~clean.nl.str.contains("random", case=False)
             & ~clean.bash.str.contains(r"\bshuf\b|sort -R|\$RANDOM|/dev/u?random"))
clean = keep(clean, "wrong scope",
             ~(clean.nl.str.contains(r"current (?:directory|folder)", case=False)
               & clean.bash.str.contains(r"find\s+(?:/|~)(?:\s|$)")))
clean = keep(clean, "catastrophic", ~clean.bash.str.contains(CATASTROPHIC))
clean = keep(clean, "test leakage",
             ~clean.nl.map(normalise).isin(test_request_set)
             & ~clean.bash.map(normalise).isin(test_command_set)
             & ~clean.nl.map(near_test_request))
clean = keep(clean, "duplicates", ~clean.nl.map(normalise).duplicated())
clean = keep(clean, "invalid syntax", clean.bash.map(valid_bash))

clean = clean.assign(utility=clean.bash.map(utility))
by_tool = clean.groupby("utility").source.agg(["size", "nunique"])
common = by_tool[(by_tool["size"] >= MIN_PER_UTILITY) | (by_tool["nunique"] >= 2)].index
clean = keep(clean, "rare tools", clean.utility.isin(common))
clean = clean.sample(frac=1, random_state=SEED)          # shuffle, so the cap keeps a random subset
cap = int(MAX_UTILITY_SHARE * len(clean))
clean = keep(clean, "tool cap", clean.groupby("utility").cumcount() < cap).reset_index(drop=True)

print(pd.DataFrame(report).to_string(index=False))
print()
print(clean.source.value_counts().to_string())
print(f"\n{clean.utility.nunique()} tools; most common:", clean.utility.value_counts().head(8).to_dict())

### Split and save

Validation holds out whole commands: every request for a held-out command
goes to validation, so paraphrases of one command never sit on both sides.
The exact files used for training are saved, with a manifest, because
`bash -n` can differ by a pair or two between bash versions.

In [ ]:
command_key = clean.bash.map(normalise)
held_out = set(command_key.drop_duplicates().sample(VAL_COMMANDS, random_state=SEED))
val_df, train_df = clean[command_key.isin(held_out)], clean[~command_key.isin(held_out)]
if MAX_TRAIN:
    train_df = train_df.head(MAX_TRAIN)

def to_messages(df):
    return [
        {"messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row.nl},
            {"role": "assistant", "content": row.bash},
        ]}
        for row in df.itertuples()
    ]

train_records, val_records = to_messages(train_df), to_messages(val_df)

train_df.to_csv(OUTPUT_DIR / "train.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "val.csv", index=False)
manifest = {
    "sources": {name: {"dataset": repo, "revision": revision} for name, (repo, revision) in SOURCES.items()},
    "filters": report,
    "train_rows": len(train_df),
    "val_rows": len(val_df),
    "train_rows_by_source": train_df.source.value_counts().to_dict(),
    "train_sha256": hashlib.sha256((OUTPUT_DIR / "train.csv").read_bytes()).hexdigest(),
    "bash": subprocess.run(["bash", "--version"], capture_output=True, text=True).stdout.splitlines()[0],
    "seed": SEED,
}
(OUTPUT_DIR / "data_manifest.json").write_text(json.dumps(manifest, indent=2))

steps_per_epoch = len(train_df) // (BATCH_SIZE * GRAD_ACCUM)
print(f"train {len(train_df):,}  val {len(val_df):,}  steps per epoch {steps_per_epoch:,}")
print(f"estimated training time on a T4: {len(train_df) * EPOCHS / 10_000 * 14.8 / 60:.1f} hours")

## 4. Fine-tuning

QLoRA with Unsloth: the base model is loaded in 4-bit, adapters are trained on
every attention and MLP projection, and the loss covers the assistant's reply
only, so the model learns to produce commands and not to repeat the prompt.

The model is evaluated and checkpointed every 250 steps, training stops early
if validation loss stops improving, and the checkpoint with the lowest
validation loss is the one saved and exported.

In [ ]:
from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

run_dir = OUTPUT_DIR / NAME
checkpoint_dir = run_dir / "checkpoints"
run_dir.mkdir(exist_ok=True)

def load(model_name):
    # The T4 has no bfloat16 support, so the model is loaded in float16 to match fp16 training.
    return FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        dtype=None if is_bfloat16_supported() else torch.float16,
    )

model, tokenizer = load(BASE_MODEL)
model = FastLanguageModel.get_peft_model(
    model,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    **LORA,
)

def render(records):
    return Dataset.from_dict(
        {"text": [tokenizer.apply_chat_template(r["messages"], tokenize=False) for r in records]}
    )

# TRL renamed two arguments across releases; use whichever this install accepts.
length_arg = "max_length" if "max_length" in inspect.signature(SFTConfig).parameters else "max_seq_length"
tokenizer_arg = ("processing_class" if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters
                 else "tokenizer")

trainer = SFTTrainer(
    model=model,
    train_dataset=render(train_records),
    eval_dataset=render(val_records),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    args=SFTConfig(
        dataset_text_field="text",
        output_dir=str(checkpoint_dir),
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="linear",
        warmup_steps=round(WARMUP_SHARE * EPOCHS * steps_per_epoch),
        weight_decay=0.01,
        optim="adamw_8bit",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=EVAL_EVERY,
        save_strategy="steps",
        save_steps=EVAL_EVERY,
        save_total_limit=2,                  # the best and the latest checkpoint
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        seed=SEED,
        report_to="none",
        **{length_arg: MAX_SEQ_LENGTH},
    ),
    **{tokenizer_arg: tokenizer},
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

In [ ]:
resume = any(checkpoint_dir.glob("checkpoint-*"))      # true after a disconnected session
start = time.time()
stats = trainer.train(resume_from_checkpoint=resume)
minutes = (time.time() - start) / 60

model.save_pretrained(run_dir / "lora_adapter")
tokenizer.save_pretrained(run_dir / "lora_adapter")
pd.DataFrame(trainer.state.log_history).to_csv(run_dir / "log_history.csv", index=False)

info = {
    "model": NAME,
    "base_model": BASE_MODEL,
    "train_examples": len(train_records),
    "val_examples": len(val_records),
    "epochs_planned": EPOCHS,
    "steps_run": trainer.state.global_step,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_eval_loss": round(trainer.state.best_metric, 4),
    "train_loss": round(stats.training_loss, 4),
    "train_minutes_this_session": round(minutes, 1),
    "resumed": resume,
    "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
    "gpu": torch.cuda.get_device_name(0),
    "precision": "bf16" if is_bfloat16_supported() else "fp16",
    "lora": LORA,
    "learning_rate": LEARNING_RATE,
}
(run_dir / "run_info.json").write_text(json.dumps(info, indent=2))
print(json.dumps(info, indent=2))

### Sanity check

A few validation requests, so obvious failures (empty replies, explanations
instead of commands) show up before the export. The test sets are not touched
here; they are only used on the laptop.

In [ ]:
FastLanguageModel.for_inference(model)
for row in val_df.sample(8, random_state=SEED).itertuples():
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": row.nl}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")
    output = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    reply = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    print(f"{row.nl}\n  model:     {reply}\n  reference: {row.bash}\n")

## 5. Export to GGUF

Both the fine-tuned and the untouched base model go through the same merge,
conversion and `Q4_K_M` quantisation. Comparing the fine-tuned file with a
base GGUF built by a different converter would mix two effects; exporting
both here removes that difference from the comparison.

The export needs several GB of temporary space, so it runs on the Colab disk
and only the final file is copied to Drive.

In [ ]:
LOCAL_EXPORT = Path("/content/gguf_export")

def export_gguf(model, tokenizer, label):
    work_dir = LOCAL_EXPORT / label
    target = OUTPUT_DIR / f"{NAME}-{label}.Q4_K_M.gguf"
    exported = next(work_dir.rglob("*Q4_K_M.gguf"), None)      # reuse a finished export after a failed copy
    if exported is None:
        model.save_pretrained_gguf(str(work_dir / "gguf"), tokenizer, quantization_method="q4_k_m")
        exported = next(work_dir.rglob("*Q4_K_M.gguf"))
    if shutil.disk_usage(OUTPUT_DIR).free < exported.stat().st_size:
        raise RuntimeError(f"Not enough space on Drive. Free some, then run this cell again; "
                           f"the exported file is kept at {exported}.")
    shutil.copy(exported, target)
    shutil.rmtree(work_dir)
    return str(target)

info["gguf"] = export_gguf(model, tokenizer, "finetuned")

del model, trainer
gc.collect()
torch.cuda.empty_cache()

# The base model has no adapter to merge, so it must be loaded in full precision:
# 4-bit weights cannot be converted to GGUF.
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=False, dtype=torch.float16
)
info["base_gguf"] = export_gguf(base_model, base_tokenizer, "base-export")

(run_dir / "run_info.json").write_text(json.dumps(info, indent=2))

## 6. Records

In [ ]:
versions = {pkg: importlib.metadata.version(pkg)
            for pkg in ["unsloth", "unsloth_zoo", "transformers", "trl", "peft", "datasets", "torch"]}
(OUTPUT_DIR / "versions.json").write_text(json.dumps(versions, indent=2))
print(json.dumps(versions, indent=2))

log = pd.read_csv(run_dir / "log_history.csv")
print(log.loc[log.eval_loss.notna(), ["step", "eval_loss"]].to_string(index=False))

## 7. Next step: test on the laptop

Copy from the Drive folder into the project:

| From Drive | To |
|---|---|
| `qwen2.5-coder-1.5b-finetuned.Q4_K_M.gguf`, `qwen2.5-coder-1.5b-base-export.Q4_K_M.gguf` | `models/` |
| `data_manifest.json`, `versions.json`, `train.csv`, `val.csv`, `qwen2.5-coder-1.5b/run_info.json`, `qwen2.5-coder-1.5b/log_history.csv` | `finetune/records/` |

Then compare the two models on both test sets. Delete the saved answers
first, so the new files are the ones being scored:

```
rm evaluation/*/results/answers/*.jsonl
cd evaluation/app_requests && uv run python run.py
cd ../nl2sh_alfa && uv run python run.py
```

The app keeps using the original Qwen GGUF for explanations: training on
command-only replies makes a model worse at explaining, and nothing here
changes that.